# Lesson 1 — Why NumPy exists

**The question:** Python already has lists, and lists can hold numbers. So why does a whole separate library exist just to hold numbers?

The answer is not "more functions." It is that a Python list stores numbers in a wasteful way, and for machine learning that waste is fatal.

## 1. Measure it first

Don't take my word for it. Run it.

In [ ]:
import sys
import time

import numpy as np

N = 10_000_000

a = list(range(N))
b = list(range(N))

start = time.perf_counter()
c = [x + y for x, y in zip(a, b)]
list_time = time.perf_counter() - start

na = np.arange(N)
nb = np.arange(N)

start = time.perf_counter()
nc = na + nb
numpy_time = time.perf_counter() - start

print(f"Adding {N:,} numbers")
print(f"  Python list : {list_time:.4f} s")
print(f"  NumPy array : {numpy_time:.4f} s")
print(f"  NumPy is {list_time / numpy_time:.0f}x faster")

## 2. Why the memory difference?

A Python `int` is not 8 bytes of number. It is a full **object**: a type tag, a reference count, and *then* the number.

A NumPy array stores the type **once for the whole array**, so each element is just the raw number.

In [ ]:
print("Memory for ONE number:")
print(f"  Python int object : {sys.getsizeof(12345)} bytes")
print(f"  NumPy int64       : {np.int64(12345).nbytes} bytes")

arr = np.arange(1_000_000)
print(f"\n1,000,000 numbers as a NumPy array: {arr.nbytes / 1024**2:.1f} MB")
print("Same as a Python list:              ~34 MB")

### What a Python list really is

`[1, 2, 3]` does **not** put 1, 2, 3 side by side in memory. It stores a list of **addresses**, each pointing to a separate object elsewhere in memory.

So `a[0] + b[0]` means: read address → jump there → check the type → unwrap the number → same for b → add → **allocate a brand new object** → write the number in → store its address.

Nine steps to add two numbers. And the loop runs in the Python interpreter, which adds its own cost every iteration.

### What NumPy does

One solid block of memory. All the same type. Type recorded once. The `+` loop runs in **compiled C**: read 8 bytes, read 8 bytes, add, write 8 bytes. No type checks, no allocation per element.

The name for this idea is **vectorization** — describe the operation on the whole array, and let compiled code run the loop.

## 3. The trap

**Question:** if you write a normal Python `for` loop over a NumPy array, do you get NumPy speed?

Guess before running the next cell.

In [ ]:
N = 1_000_000
py_list = list(range(N))
np_arr = np.arange(N)

start = time.perf_counter()
for i in range(N):
    py_list[i] + 1
t_list_loop = time.perf_counter() - start

start = time.perf_counter()
for i in range(N):
    np_arr[i] + 1
t_numpy_loop = time.perf_counter() - start

start = time.perf_counter()
np_arr + 1
t_numpy_vec = time.perf_counter() - start

print(f"  Python loop over Python list : {t_list_loop:.4f} s")
print(f"  Python loop over NumPy array : {t_numpy_loop:.4f} s")
print(f"  Vectorized NumPy (no loop)   : {t_numpy_vec:.4f} s")
print()
print(f"  Looping over the array is {t_numpy_loop / t_list_loop:.1f}x SLOWER than over a list")
print(f"  Vectorized is {t_numpy_loop / t_numpy_vec:.0f}x faster than looping")

### Why looping over a NumPy array is *worse* than a list

Speed does not come from how the data is **stored**. It comes from **where the loop runs**.

If you write the `for` loop, the loop is in Python. The C code never gets to run — it only ever sees one element at a time.

And it's worse than neutral: every time you touch a single element, NumPy must take the raw bytes and **wrap them back into a Python object** so Python can use them. That's called **boxing**, and a list never has to do it.

In [ ]:
# proof of the boxing: touching one element gives you a Python-visible object
print(type(np_arr[0]))

## The analogy

You have a batch API endpoint that takes 1,000 items in one call. But you call it **1,000 times with 1 item each**.

You get no batching benefit — you pay the per-call overhead every time, plus the batch format's extra cost.

**A Python loop over a NumPy array is exactly this.** The batch endpoint only helps if you actually send the batch.

## The two things to remember

1. NumPy is fast because the **loop moves into C** — not because the data is packed. Packed data is what makes the C loop *possible*.
2. **If you see `for` next to a NumPy array, something is usually wrong.** There is nearly always a whole-array operation that does the same job.